# 2 — Génération des QCM : baseline

La baseline est un générateur extractif déterministe : une phrase du support devient la réponse, un terme TF‑IDF sert à construire une question, et trois phrases distinctes du corpus forment les distracteurs. Ce choix assure une traçabilité, mais ne remplace pas un générateur de langue naturelle ; cette limite sera discutée dans l’article.

In [1]:
from pathlib import Path
import sys, random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'requirements.txt').exists(): ROOT = ROOT.parent
if not (ROOT / 'requirements.txt').exists(): raise FileNotFoundError('Racine du projet introuvable.')
sys.path.insert(0, str(ROOT))
from src.qcm_pipeline import generate_mcqs
PASSAGES_PATH = ROOT / 'data' / 'processed' / 'passages.csv'
if not PASSAGES_PATH.exists(): raise FileNotFoundError('Exécuter d’abord 01_preparation_donnees.ipynb.')
passages = pd.read_csv(PASSAGES_PATH)
candidates = generate_mcqs(passages, limit=120)
OUT = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)
candidates.to_csv(OUT / 'qcm_candidats.csv', index=False, encoding='utf-8-sig')
print(f'{len(candidates)} QCM candidats générés (baseline non filtrée).')
print('Répartition par support :')
display(candidates.groupby('source_doc').size().rename('QCM').to_frame())
display(candidates[['question_id', 'question', 'options', 'correct_index', 'source_doc', 'source_page']].head(5))

120 QCM candidats générés (baseline non filtrée).
Répartition par support :


,QCM
source_doc,
01_vision_ordinateur_ch1.pdf,114
02_vision_ordinateur_ch2.pdf,6


,question_id,question,options,correct_index,source_doc,source_page
0,Q0001,"Selon le support, quelle affirmation concernan...","[""1.Le cerveau devine d’abord la couleur de la...",3,01_vision_ordinateur_ch1.pdf,4
1,Q0002,"Selon le support, quelle affirmation concernan...","[""Chaque cerveau estime la lumière différemmen...",2,01_vision_ordinateur_ch1.pdf,4
2,Q0003,"Selon le support, quelle affirmation concernan...","[""1.Le cerveau devine d’abord la couleur de la...",1,01_vision_ordinateur_ch1.pdf,4
3,Q0004,"Selon le support, quelle affirmation concernan...","[""Alex Krizhevsky, et son mentor Geoffrey Hint...",3,01_vision_ordinateur_ch1.pdf,5
4,Q0005,"Selon le support, quelle affirmation concernan...","[""Différence fondamentale entre vision humaine...",0,01_vision_ordinateur_ch1.pdf,5


## Vérification du format et exemples

Chaque item doit avoir quatre choix, une seule position de réponse de référence et une citation de page. La validité sémantique sera évaluée au notebook suivant puis par des évaluateurs humains.

In [2]:
import json
assert candidates['question_id'].is_unique
assert candidates['correct_index'].between(0, 3).all()
assert candidates['options'].map(lambda x: len(json.loads(x))).eq(4).all()
assert candidates['question'].str.len().gt(15).all()
print('Contrôles structurels réussis : 4 options, une réponse de référence et provenance pour chaque QCM.')
sample = candidates.iloc[0]
print(f"Exemple {sample.question_id} — {sample.source_doc}, page {sample.source_page}")
print(sample.question)
for i, option in enumerate(json.loads(sample.options)):
    print(f"{chr(65+i)}. {option}")
print('Réponse de référence :', chr(65 + int(sample.correct_index)))

Contrôles structurels réussis : 4 options, une réponse de référence et provenance pour chaque QCM.
Exemple Q0001 — 01_vision_ordinateur_ch1.pdf, page 4
Selon le support, quelle affirmation concernant « surface objet » est correcte ?
A. 1.Le cerveau devine d’abord la couleur de la source lumineuse 2.Il effectue un jugement relatif.
B. Plus une image numérique possède une grille de pixels dense (haute résolution), plus elle se rapproche visuellement de la fluidité d'une image analogique pour l'œil humain.
C. """ Créer une image en couleur ou en niveaux de gris avec des valeurs aléatoires.
D. L’œil humain ne perçoit pas la couleur comme une propriété absolue de la surface d’un objet.
Réponse de référence : D
